# Multi-Audience Explainer

Day 1 project: scrape a webpage and explain the same content three times, once per audience, using a different system prompt each time - a curious 8-year-old, a high school student, and a subject-matter expert.

Same scraped content, three system prompts, three very different explanations. It's a quick way to see how much the system prompt alone shapes tone and depth.

In [ ]:
# imports

import requests
from bs4 import BeautifulSoup
from dotenv import load_dotenv
from openai import OpenAI
from IPython.display import Markdown, display

In [ ]:
# Load environment variables and connect to OpenAI

load_dotenv(override=True)
openai = OpenAI()

## Scrape the page

Same approach as `scraper.py` from Day 1: fetch the page, strip scripts/styles/images, and keep the first 2,000 characters of body text.

In [ ]:
headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/117.0.0.0 Safari/537.36"
}

def fetch_website_contents(url):
    response = requests.get(url, headers=headers)
    soup = BeautifulSoup(response.content, "html.parser")
    title = soup.title.string if soup.title else "No title found"
    for irrelevant in soup.body(["script", "style", "img", "input"]):
        irrelevant.decompose()
    text = soup.body.get_text(separator="\n", strip=True)
    return (title + "\n\n" + text)[:2_000]

## Define the audiences

Each audience is just a different system prompt. Add your own to the dictionary to try more.

In [ ]:
audiences = {
    "An 8-year-old": "You explain things to an 8-year-old. Use very simple words, short sentences, and a fun analogy. Avoid jargon completely.",
    "A high school student": "You explain things to a high school student. Use clear language, define any technical terms you use, and keep it engaging.",
    "A subject-matter expert": "You explain things to a subject-matter expert. Be precise and technical, and call out anything noteworthy or unusual.",
}

def user_prompt_for(website_text):
    return f"Here is the content of a webpage. Explain what it is about.\n\n{website_text}"

## Generate an explanation per audience

In [ ]:
def explain_for(audience, website_text):
    messages = [
        {"role": "system", "content": audiences[audience]},
        {"role": "user", "content": user_prompt_for(website_text)},
    ]
    response = openai.chat.completions.create(model="gpt-4.1-mini", messages=messages)
    return response.choices[0].message.content

def explain_url(url):
    website_text = fetch_website_contents(url)
    for audience in audiences:
        display(Markdown(f"### Explained for: {audience}"))
        display(Markdown(explain_for(audience, website_text)))

## Try it out

In [ ]:
explain_url("https://edwarddonner.com")